## 井点数据预处理


In [ ]:
import numpy as np
import pandas as pd

In [ ]:
# 读取数据
file_well = "../data/target/well_horizon.xlsx"
data_well = pd.read_excel(file_well, sheet_name="Sand Thickness")

print("原始数据形状：", data_well.shape)
print("原始数据前5行：")
print(data_well.head())

In [ ]:
# 获取列名 - 根据位置确定
xyz_columns = data_well.columns[0:3].tolist()  # XYZ坐标前3列
surface_column = data_well.columns[9]  # 层位名在第10列
well_column = data_well.columns[10]  # 井名在第11列
sand_thickness_column = data_well.columns[27]  # 砂厚在第28列
count_column = data_well.columns[28]  # 重复计数在第29列

print(f"\nXYZ坐标列: {xyz_columns}")
print(f"层位名列: {surface_column}")
print(f"井名列: {well_column}")
print(f"砂厚列: {sand_thickness_column}")
print(f"重复计数列: {count_column}")

In [ ]:
# 2. 设置要删除的层位列表（默认只有P0，可手动添加其他）
horizons_to_delete = ["P0(H83D)"]

# 3. 设置要删除的井点列表（默认为空，可手动添加）
wells_to_delete = []

In [ ]:
# 开始数据处理
print(f"\n=== 开始数据处理 ===")

# 删除指定层位
data_filtered = data_well.copy()
if horizons_to_delete:
    before_count = len(data_filtered)
    data_filtered = data_filtered[~data_filtered[surface_column].isin(horizons_to_delete)]
    after_count = len(data_filtered)
    print(f"删除层位 {horizons_to_delete}，删除了 {before_count - after_count} 行数据")

# 删除指定井点
if wells_to_delete:
    before_count = len(data_filtered)
    data_filtered = data_filtered[~data_filtered[well_column].isin(wells_to_delete)]
    after_count = len(data_filtered)
    print(f"删除井点 {wells_to_delete}，删除了 {before_count - after_count} 行数据")


In [ ]:
# 4. 删除Sand Thickness=-999的行
missing_value_mask = data_filtered[sand_thickness_column] == -999
missing_count = missing_value_mask.sum()

if missing_count > 0:
    print(f"发现 {missing_count} 个砂厚值为-999的数据，将直接删除")

    # 显示一些要删除的数据示例
    missing_data = data_filtered[missing_value_mask]
    print("要删除的数据示例（前5行）：")
    for idx, row in missing_data.head().iterrows():
        print(f"  井: {row[well_column]}, 层位: {row[surface_column]}, 砂厚: {row[sand_thickness_column]}")

    # 删除砂厚为-999的行
    data_filtered = data_filtered[~missing_value_mask].reset_index(drop=True)
    print(f"已删除 {missing_count} 行砂厚为-999的数据，剩余数据 {len(data_filtered)} 行")
else:
    print("未发现砂厚值为-999的数据")

In [ ]:
# 5. 处理重复计数大于1的情况，只保留砂厚值最大的一行
duplicate_mask = data_filtered[count_column] > 1
duplicate_count = duplicate_mask.sum()

if duplicate_count > 0:
    print(f"\n发现 {duplicate_count} 行重复计数大于1的数据")

    # 对于重复计数大于1的数据，按井名和层位分组，保留砂厚最大的一行
    duplicates = data_filtered[duplicate_mask]
    non_duplicates = data_filtered[~duplicate_mask]

    # 显示一些重复数据的例子
    print("重复数据示例：")
    for (well, surface), group in duplicates.groupby([well_column, surface_column]):
        if len(group) > 0:
            print(
                f"  井 {well}, 层位 {surface}: {len(group)} 行数据，砂厚范围 {group[sand_thickness_column].min():.2f} - {group[sand_thickness_column].max():.2f}"
            )
            break

    # 保留砂厚最大的一行
    max_thickness_duplicates = duplicates.loc[
        duplicates.groupby([well_column, surface_column])[sand_thickness_column].idxmax()
    ]

    # 合并非重复数据和处理后的重复数据
    data_processed = pd.concat([non_duplicates, max_thickness_duplicates], ignore_index=True)

    removed_count = len(data_filtered) - len(data_processed)
    print(f"从重复数据中移除了 {removed_count} 行，保留了砂厚最大的行")
else:
    data_processed = data_filtered
    print("没有发现重复计数大于1的数据")

In [ ]:
print(f"\n=== 离群值处理 ===")

# 计算离群值阈值（使用IQR方法，偏保守）
Q1 = data_processed[sand_thickness_column].quantile(0.25)
Q3 = data_processed[sand_thickness_column].quantile(0.75)
IQR = Q3 - Q1

# 使用3.0倍IQR作为离群值标准
lower_bound = Q1 - 3.0 * IQR
upper_bound = Q3 + 3.0 * IQR

print(f"砂厚统计信息：")
print(f"Q1 (25%分位数): {Q1:.2f} 米")
print(f"Q3 (75%分位数): {Q3:.2f} 米")
print(f"IQR: {IQR:.2f} 米")
print(f"离群值阈值: {lower_bound:.2f} - {upper_bound:.2f} 米")

# 识别离群值
outliers_mask = (data_processed[sand_thickness_column] < lower_bound) | (
    data_processed[sand_thickness_column] > upper_bound
)
outliers_count = outliers_mask.sum()

if outliers_count > 0:
    print(f"\n发现 {outliers_count} 个离群值:")
    outliers = data_processed[outliers_mask]

    # 显示离群值信息
    for idx, row in outliers.iterrows():
        print(f"  井: {row[well_column]}, 层位: {row[surface_column]}, 砂厚: {row[sand_thickness_column]:.2f} 米")

    # 删除离群值
    data_processed = data_processed[~outliers_mask].reset_index(drop=True)

    print(f"\n已删除 {outliers_count} 个离群值，剩余数据 {len(data_processed)} 行")
else:
    print("未发现离群值")

In [ ]:
# 最终统计
print(f"\n=== 处理结果统计 ===")
print(f"原始数据: {len(data_well)} 行")
print(f"处理后数据: {len(data_processed)} 行")
print(f"共有 {len(data_processed[well_column].unique())} 个不同的井点")
print(f"共有 {len(data_processed[surface_column].unique())} 个不同的层位")

# 统计每个井的数据量
well_counts = data_processed[well_column].value_counts()
print(f"\n每个井的数据量统计:")
print(f"最多: {well_counts.max()} 行 (井: {well_counts.idxmax()})")
print(f"最少: {well_counts.min()} 行 (井: {well_counts.idxmin()})")
print(f"平均: {well_counts.mean():.1f} 行")

# 统计砂厚分布（处理离群值后）
sand_thickness_stats = data_processed[sand_thickness_column].describe()
print(f"\n砂厚分布统计（处理离群值后）:")
print(sand_thickness_stats)

zero_thickness_count = (data_processed[sand_thickness_column] == 0).sum()
positive_thickness_count = (data_processed[sand_thickness_column] > 0).sum()
print(f"砂厚为0的样本: {zero_thickness_count} 个 ({zero_thickness_count / len(data_processed) * 100:.1f}%)")
print(f"砂厚大于0的样本: {positive_thickness_count} 个 ({positive_thickness_count / len(data_processed) * 100:.1f}%)")

# 显示处理后的前几行数据
print(f"\n处理后数据前5行:")
print(data_processed.head())

In [ ]:
# 6. 保存处理后的数据
output_file = "../data/target/well_horizon_processed.xlsx"
data_processed.to_excel(output_file, index=False)
print(f"\n数据已保存到: {output_file}")